In [1]:
import subprocess
from urllib.parse import quote_plus
from sqlalchemy import create_engine, inspect
from datetime import datetime
import os
import pymysql
import gzip


In [1]:
pip install sqlalchemy pymysql

  Using cached sqlalchemy-2.0.46-cp314-cp314-win_amd64.whl.metadata (9.8 kB)
  Using cached pymysql-1.1.2-py3-none-any.whl.metadata (4.3 kB)
  Using cached greenlet-3.3.1-cp314-cp314-win_amd64.whl.metadata (3.8 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached sqlalchemy-2.0.46-cp314-cp314-win_amd64.whl (2.1 MB)
Using cached pymysql-1.1.2-py3-none-any.whl (45 kB)
Using cached greenlet-3.3.1-cp314-cp314-win_amd64.whl (228 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)

   ---------- ----------------------------- 1/4 [pymysql]
   -------------------- ------------------- 2/4 [greenlet]
   -------------------- ------------------- 2/4 [greenlet]
   ------------------------------ --------- 3/4 [sqlalchemy]
   ------------------------------ --------- 3/4 [sqlalchemy]
   ------------------------------ --------- 3/4 [sqlalchemy]
   ------------------------------ --------- 3/4 [sqlalchemy]
   ------------------------------ --------- 3

In [2]:
# Your provided configuration
SOURCE_DB_CONFIG = {
    "user": "root",
    "password": "mysql",
    "host": "localhost",
    "port": "3306",
    "database": "obeta_db"
}
BACKUP_DIR = "./backups"

In [14]:
# Helper to create SQLAlchemy engine
def get_db_engine(config):
    password = quote_plus(config['password'])
    connection_str = (
        f"mysql+pymysql://{config['user']}:{password}"
        f"@{config['host']}:{config['port']}/{config['database']}"
    )
    return create_engine(connection_str)

# Optional: Load & inspect schema
def load_schema(engine):
    inspector = inspect(engine)

    tables = inspector.get_table_names()
    schema = {}

    for table in tables:
        schema[table] = inspector.get_columns(table)

    return schema

In [15]:
source_engine = get_db_engine(SOURCE_DB_CONFIG)
source_schema = load_schema(source_engine)

In [19]:
def create_backup(config, backup_dir="backups", schema_only=False,compress=True):
    os.makedirs(backup_dir, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    extension = ".sql.gz" if compress else ".sql"
    backup_file = os.path.join(
        backup_dir,
        f"{config['database']}_{timestamp}.sql"
    )
    # Build mysqldump command
    dump_command = [
        "mysqldump",
        "-h", config["host"],
        "-P", config["port"],
        "-u", config["user"],
        f"--password={config['password']}",
        config["database"]
    ]

    if schema_only:
        dump_command.insert(1, "--no-data")

    if compress:
        # Start mysqldump process
        dump_proc = subprocess.Popen(
            dump_command,
            stdout=subprocess.PIPE
        )

        # Pipe output into gzip
        with open(backup_file, "wb") as f:
            gzip_proc = subprocess.Popen(
                ["gzip"],
                stdin=dump_proc.stdout,
                stdout=f
            )

        dump_proc.stdout.close()
        gzip_proc.communicate()

        if dump_proc.returncode not in (0, None):
            raise RuntimeError("mysqldump failed")

    else:
        # No compression
        with open(backup_file, "w") as f:
            subprocess.run(dump_command, stdout=f, check=True)

    return backup_file

In [21]:
# Example usage
backup_path = create_backup(SOURCE_DB_CONFIG, backup_dir=BACKUP_DIR, compress=False)
#print(f"Backup created at: {backup_path}")